In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


# ===============================
# 1. ĐỌC DỮ LIỆU
# ===============================

df = pd.read_csv("favorite_music_dataset.csv")

print("===== 5 DÒNG ĐẦU DATASET =====")
print(df.head())

print("\n===== THÔNG TIN DATASET =====")
print(df.info())

print("\n===== CÁC THỂ LOẠI NHẠC BAN ĐẦU =====")
print(df["Genre"].value_counts())


# ===============================
# 2. GOM NHÓM THỂ LOẠI NHẠC
# ===============================
# Đề bài yêu cầu output: Pop / Rock / EDM
# Nên mình gom các genre nhỏ về 3 nhóm chính.

def map_genre_to_group(genre):
    genre = str(genre).lower()

    if "electronic" in genre or "edm" in genre or "synth" in genre or "electropop" in genre:
        return "EDM"

    elif "rock" in genre or "alternative" in genre or "indie" in genre:
        return "Rock"

    elif "pop" in genre or "soul" in genre or "r&b" in genre or "hip hop" in genre:
        return "Pop"

    else:
        return "Pop"


df["Genre_Group"] = df["Genre"].apply(map_genre_to_group)

print("\n===== GENRE SAU KHI GOM NHÓM =====")
print(df["Genre_Group"].value_counts())


# ===============================
# 3. CHỌN INPUT VÀ OUTPUT
# ===============================

X = df[[
    "Song_Title",
    "Artist",
    "Release_Year",
    "Duration_Minutes",
    "Platform"
]]

y = df["Genre_Group"]


# ===============================
# 4. CHIA TRAIN / TEST
# ===============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ===============================
# 5. TIỀN XỬ LÝ DỮ LIỆU
# ===============================

categorical_features = [
    "Song_Title",
    "Artist",
    "Platform"
]

numeric_features = [
    "Release_Year",
    "Duration_Minutes"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)


# ===============================
# 6. TẠO MÔ HÌNH PERCEPTRON
# ===============================

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", Perceptron(
        max_iter=1000,
        eta0=0.01,
        random_state=42
    ))
])


# ===============================
# 7. HUẤN LUYỆN MÔ HÌNH
# ===============================

model.fit(X_train, y_train)


# ===============================
# 8. DỰ ĐOÁN
# ===============================

y_pred = model.predict(X_test)


# ===============================
# 9. ĐÁNH GIÁ MÔ HÌNH
# ===============================

accuracy = accuracy_score(y_test, y_pred)

print("\n===== KẾT QUẢ MÔ HÌNH PERCEPTRON =====")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(y_test, y_pred))


# ===============================
# 10. IN THỬ KẾT QUẢ DỰ ĐOÁN
# ===============================

result_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\n===== 10 KẾT QUẢ DỰ ĐOÁN ĐẦU TIÊN =====")
print(result_df.head(10))


# ===============================
# 11. DỰ ĐOÁN MỘT MẪU MỚI
# ===============================

new_music = pd.DataFrame([{
    "Song_Title": "Faded",
    "Artist": "Alan Walker",
    "Release_Year": 2024,
    "Duration_Minutes": 4.02,
    "Platform": "Spotify"
}])

prediction = model.predict(new_music)[0]

print("\n===== DỰ ĐOÁN MẪU NHẠC MỚI =====")
print("Thể loại nhạc dự đoán:", prediction)

===== 5 DÒNG ĐẦU DATASET =====
          Song_Title       Artist       Genre  Release_Year  Duration_Minutes  \
0              Faded  Alan Walker  Electronic          2024              4.02   
1    Blinding Lights   The Weeknd   Synth-pop          2018              4.45   
2              Faded  Alan Walker  Electronic          2024              4.86   
3              Faded  Alan Walker  Electronic          2012              4.92   
4  Bohemian Rhapsody        Queen        Rock          2023              3.90   

  Listened_Date     Platform  
0    2024-02-10  Apple Music  
1    2024-05-30     Zing MP3  
2    2024-05-07     Zing MP3  
3    2024-03-19      YouTube  
4    2024-01-16  Apple Music  

===== THÔNG TIN DATASET =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Song_Title        100 non-null    object 
 1   Artist          

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import ipywidgets as widgets
from IPython.display import display, clear_output

data = {
    "Age": [
        15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27,
        16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
        14, 15, 16, 17, 18, 19, 20, 21, 22, 23,
        24, 25, 26, 27, 28, 29, 30, 31, 32, 33
    ],

    "Hours_Listening": [
        1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 2.0, 3.0, 4.5,
        0.5, 1.0, 1.5, 2.0, 2.5, 1.0, 1.5, 2.0, 2.5, 3.0,
        3.0, 3.5, 4.0, 4.5, 5.0, 4.0, 5.5, 6.0, 5.0, 6.5,
        2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5,
        1.0, 1.5, 2.0, 2.5, 3.0, 2.0, 2.5, 3.0, 3.5, 4.0
    ],

    "Listening_Habit": [
        1, 1, 2, 2, 1, 2, 1, 1, 2, 2,
        0, 0, 0, 0, 0, 0, 1, 0, 1, 0,
        2, 2, 2, 1, 2, 2, 1, 2, 2, 1,
        1, 2, 1, 2, 2, 1, 2, 1, 2, 2,
        0, 0, 0, 1, 0, 1, 0, 1, 0, 1
    ],

    "Favorite_Music": [
        "Pop", "Pop", "Pop", "Pop", "Pop", "Pop", "Pop", "Pop", "Pop", "Pop",
        "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock",
        "EDM", "EDM", "EDM", "EDM", "EDM", "EDM", "EDM", "EDM", "EDM", "EDM",
        "EDM", "EDM", "Pop", "Pop", "EDM", "EDM", "EDM", "EDM", "EDM", "EDM",
        "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock", "Rock"
    ]
}

df = pd.DataFrame(data)

print("===== DATASET =====")
display(df.head())

print("\nKích thước dataset:", df.shape)

print("\nSố lượng từng nhóm nhạc:")
print(df["Favorite_Music"].value_counts())

X = df[["Age", "Hours_Listening", "Listening_Habit"]]
y = df["Favorite_Music"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = Perceptron(
    max_iter=1000,
    eta0=0.01,
    random_state=42
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print("===== KẾT QUẢ MÔ HÌNH PERCEPTRON =====")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n===== CONFUSION MATRIX =====")
print(confusion_matrix(y_test, y_pred))

print("\n===== CLASSIFICATION REPORT =====")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    zero_division=0
))


age_input = widgets.IntSlider(
    value=20,
    min=10,
    max=60,
    step=1,
    description="Age:",
    style={"description_width": "initial"}
)

hours_input = widgets.FloatSlider(
    value=2.0,
    min=0.0,
    max=10.0,
    step=0.5,
    description="Hours listening/day:",
    style={"description_width": "initial"}
)

habit_input = widgets.Dropdown(
    options=[
        ("Radio", 0),
        ("Spotify", 1),
        ("YouTube", 2)
    ],
    value=1,
    description="Listening habit:",
    style={"description_width": "initial"}
)

predict_button = widgets.Button(
    description="Predict",
    button_style="success",
    tooltip="Click to predict favorite music"
)

output_area = widgets.Output()


def predict_music(button):
    with output_area:
        clear_output()

        age = age_input.value
        hours = hours_input.value
        habit = habit_input.value

        new_data = pd.DataFrame([{
            "Age": age,
            "Hours_Listening": hours,
            "Listening_Habit": habit
        }])

        new_data_scaled = scaler.transform(new_data)

        prediction_encoded = model.predict(new_data_scaled)[0]

        prediction_label = label_encoder.inverse_transform([prediction_encoded])[0]

        print("===== PREDICTION RESULT =====")
        print(f"Age: {age}")
        print(f"Hours listening/day: {hours}")

        if habit == 0:
            habit_text = "Radio"
        elif habit == 1:
            habit_text = "Spotify"
        else:
            habit_text = "YouTube"

        print(f"Listening habit: {habit_text}")
        print("--------------------------------")
        print(f"Predicted favorite music: {prediction_label}")


predict_button.on_click(predict_music)

display(widgets.HTML("<h2>Favorite Music Prediction App Using Perceptron</h2>"))
display(age_input)
display(hours_input)
display(habit_input)
display(predict_button)
display(output_area)

===== DATASET =====


,Age,Hours_Listening,Listening_Habit,Favorite_Music
0,15,1.0,1,Pop
1,16,1.5,1,Pop
2,17,2.0,2,Pop
3,18,2.5,2,Pop
4,19,3.0,1,Pop



Kích thước dataset: (50, 4)

Số lượng từng nhóm nhạc:
Favorite_Music
Rock    20
EDM     18
Pop     12
Name: count, dtype: int64
===== KẾT QUẢ MÔ HÌNH PERCEPTRON =====
Accuracy: 70.00%

===== CONFUSION MATRIX =====
[[3 0 1]
 [0 0 2]
 [0 0 4]]

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

         EDM       1.00      0.75      0.86         4
         Pop       0.00      0.00      0.00         2
        Rock       0.57      1.00      0.73         4

    accuracy                           0.70        10
   macro avg       0.52      0.58      0.53        10
weighted avg       0.63      0.70      0.63        10



HTML(value='<h2>Favorite Music Prediction App Using Perceptron</h2>')

IntSlider(value=20, description='Age:', max=60, min=10, style=SliderStyle(description_width='initial'))

FloatSlider(value=2.0, description='Hours listening/day:', max=10.0, step=0.5, style=SliderStyle(description_w…

Dropdown(description='Listening habit:', index=1, options=(('Radio', 0), ('Spotify', 1), ('YouTube', 2)), styl…

Button(button_style='success', description='Predict', style=ButtonStyle(), tooltip='Click to predict favorite …

Output()